In [ ]:
import kagglehub
import pandas as pd
import os

# 1. Dataset download karein
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

# 2. Folder ke andar ki files check karein
files_in_folder = os.listdir(path)
print("Files inside the folder:", files_in_folder)

# 3. CSV file ko load karein
# Is dataset mein aamtaur par 'IMDB Dataset.csv' naam ki file hoti hai
csv_file_path = os.path.join(path, "IMDB Dataset.csv") 

# DataFrame mein read karein
original_df = pd.read_csv(csv_file_path)

# 4. Data check karein
print("\nDataset ke pehle 5 rows:")
print(original_df.head())

In [ ]:
df = original_df.copy(deep=True)

In [ ]:
df.info

In [ ]:
df.describe()

#### Descriptive Statistics

In [ ]:
# Summary Stats
df["review_char_len"] = df["review"].apply(len)
df["review_word_len"] = df["review"].apply(lambda x : len(x.split()))

# Descriptive Stats
df[["review_char_len" , "review_word_len"]].describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
# Sentiment distribution
sns . countplot (x='sentiment', data=df)
plt.title("Distribution of Sentiments")
plt.show()

In [ ]:
# Histogram of review word lengths
sns.histplot(data=df, x='review_word_len', bins=50, hue='sentiment', kde=True)
plt.title("Review Length Distribution (Words)")
plt.show()

In [ ]:
# KDE (Density Plot) of Review Lengths
plt.figure(figsize=(8,6))
sns.kdeplot(df[df['sentiment'] == 'positive']['review_word_len'], label="Positive", fill=True, alpha=0.5)
sns.kdeplot(df[df['sentiment'] == 'negative']['review_word_len'], label="Negative", fill=True, alpha=0.5)
plt.title("Density Plot of Review Lengths by Sentiment")
plt.xlabel("Review Length (words)")
plt.ylabel("Density")
plt.legend()
plt. show()

In [ ]:
# Boxplot of Review Length by Sentiment
plt.figure(figsize=(8,6))
sns.boxplot(x="sentiment", y="review_word_len", data=df, palette="Set2", hue="sentiment")
plt.title("Boxplot of Review Lengths by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Review Length (words)")
plt.show()

In [ ]:
# Calculate Q1, Q3, and IQR
Q1 = df['review_word_len'].quantile(0.25)
Q3 = df['review_word_len'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

In [ ]:
outliers = df[(df['review_word_len'] < lower_bound) | (df['review_word_len'] > upper_bound)]
print("Number of outliers:", outliers. shape[0])

In [ ]:
df_clean_from_iqr = df[(df['review_word_len'] >= lower_bound) & (df['review_word_len'] <= upper_bound)]
print("Cleaned Data :", df_clean_from_iqr.shape[0])

In [ ]:
# Boxplot of Review Length by Sentiment
plt.figure(figsize=(8,6))
sns.boxplot(x="sentiment", y="review_word_len", data=df_clean_from_iqr, palette="Set2", hue="sentiment")
plt.title("Boxplot of Review Lengths by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Review Length (words)")
plt. show()

In [ ]:
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess_text(text):
    # Lowercase & remove non-alphabetic characters
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [w for w in words if w not in stop_words]

In [ ]:
# Separate reviews by sentiment
pos_reviews = df[df['sentiment'] == "positive"]['review' ]. apply(preprocess_text)
neg_reviews = df[df['sentiment'] == "negative"]['review'].apply(preprocess_text)

In [ ]:
# Flatten lists
pos_words = [word for review in pos_reviews for word in review]
neg_words = [word for review in neg_reviews for word in review]


# Get most common words
pos_common = Counter(pos_words).most_common(20)
neg_common = Counter(neg_words) .most_common (20)

print("Top 20 Positive Words:", pos_common)
print("Top 20 Negative Words:", neg_common)

In [ ]:
from sklearn. feature_extraction.text import CountVectorizer

def get_top_ngrams(words_list, ngram_range=(2,2), top_n=10):
    vec = CountVectorizer(ngram_range=ngram_range)
    bag = vec.fit_transform([' '.join(words_list)])
    sum_words = bag. sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items() ]
    sorted_words = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return sorted_words[:top_n]

In [ ]:
print("Top Positive Bigrams:", get_top_ngrams(pos_words, (2,2)))
print("Top Negative Bigrams:", get_top_ngrams(neg_words, (2,2)))

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

# Get most common words
pos_common_300 = Counter(pos_words).most_common(300)
neg_common_300 = Counter(neg_words).most_common(300)

pos_word_dict = dict(pos_common_300)
neg_word_dict = dict(neg_common_300)

# Generate a word cloud for positive words
pos_wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(pos_word_dict)

# Generate a word cloud for negative words
neg_wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(neg_word_dict)

# Plot the word clouds
plt.figure(figsize=(20, 8))

plt.subplot(2, 1, 1)
plt. imshow(pos_wordcloud, interpolation='bilinear')
plt.title('Most Common Positive Words')
plt.axis('off')

plt.subplot(2, 1,2)
plt.imshow(neg_wordcloud, interpolation='bilinear')
plt.title('Most Common Negative Words')
plt.axis('off')

plt.tight_layout
()

plt.show()